<a href="https://colab.research.google.com/github/hammadshub/moe-code-summarization/blob/main/moe_code_summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers==4.37.2 accelerate==0.27.2 peft==0.10.0 datasets evaluate rouge_score

In [2]:
from datasets import load_dataset
ds = load_dataset("code-search-net/code_search_net", "python", trust_remote_code=True)
print(ds)
print(ds["train"][0])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'code-search-net/code_search_net' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'code-search-net/code_search_net' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your sessi

DatasetDict({
    train: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 412178
    })
    test: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 22176
    })
    validation: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 23107
    })
})
{'repository_name': 'mjirik/imcut', 'func_path_in_repository': 'imcut/pycut.py', 'func_name': 'Imag

In [3]:
small_train = ds["train"].select(range(3000))
small_test = ds["test"].select(range(300))

print(small_train)
print(small_test)

Dataset({
    features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
    num_rows: 3000
})
Dataset({
    features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
    num_rows: 300
})


In [4]:
import ast

def categorize_code(code_str):
    try:
        tree = ast.parse(code_str)
        for node in ast.walk(tree):
            if isinstance(node, ast.ClassDef):
                return "class"
            if isinstance(node, ast.FunctionDef):
                return "function"
        return "script"
    except SyntaxError:
        return "script"  # fallback if code doesn't parse cleanly


In [5]:
from collections import Counter

small_train = small_train.map(lambda x: {"category": categorize_code(x["func_code_string"])})
small_test = small_test.map(lambda x: {"category": categorize_code(x["func_code_string"])})

print("Train categories:", Counter(small_train["category"]))
print("Test categories:", Counter(small_test["category"]))

Train categories: Counter({'function': 2975, 'script': 25})
Test categories: Counter({'function': 300})


In [6]:
import ast

def categorize_code(code_str):
    try:
        tree = ast.parse(code_str)
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                args = node.args.args
                if args and args[0].arg in ("self", "cls"):
                    return "method"
                else:
                    return "function"
        return "other"
    except SyntaxError:
        return "other"

small_train = small_train.map(lambda x: {"category": categorize_code(x["func_code_string"])})
small_test = small_test.map(lambda x: {"category": categorize_code(x["func_code_string"])})

from collections import Counter
print("Train categories:", Counter(small_train["category"]))
print("Test categories:", Counter(small_test["category"]))

Train categories: Counter({'method': 1677, 'function': 1298, 'other': 25})
Test categories: Counter({'method': 175, 'function': 125})


In [7]:
small_train = small_train.filter(lambda x: x["category"] in ("method", "function"))
small_test = small_test.filter(lambda x: x["category"] in ("method", "function"))

print("Train categories:", Counter(small_train["category"]))
print("Test categories:", Counter(small_test["category"]))

Train categories: Counter({'method': 1677, 'function': 1298})
Test categories: Counter({'method': 175, 'function': 125})


In [8]:
from transformers import AutoTokenizer

MODEL_NAME = "Salesforce/codet5p-220m"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 64


def preprocess(batch):
    inputs = tokenizer(
        batch["func_code_string"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length"
    )

    targets = tokenizer(
        text_target=batch["func_documentation_string"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length"
    )

    inputs["labels"] = targets["input_ids"]

    return inputs


train_tokenized = small_train.map(
    preprocess,
    batched=True,
    remove_columns=small_train.column_names
)

test_tokenized = small_test.map(
    preprocess,
    batched=True,
    remove_columns=small_test.column_names
)

print("Tokenization completed successfully!")
print(train_tokenized)
print(test_tokenized)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Tokenization completed successfully!
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2975
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 300
})


In [9]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./baseline_model",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=5e-5,
    evaluation_strategy="epoch",   # renamed from eval_strategy
    save_strategy="epoch",
    predict_with_generate=True,
    logging_steps=50,
    fp16=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer
)

trainer.train()
trainer.save_model("./baseline_model_final")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss
1,0.480700,0.282459
2,0.048400,0.052224
3,0.028000,0.060382


Checkpoint destination directory ./baseline_model/checkpoint-372 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./baseline_model/checkpoint-744 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./baseline_model/checkpoint-1116 already exists and is non-empty.Saving will proceed but saved results may be invalid.


In [10]:
import numpy as np
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

predictions_output = trainer.predict(test_tokenized, max_length=64)
pred_ids = predictions_output.predictions
label_ids = predictions_output.label_ids

print("pred_ids shape:", pred_ids.shape)
print("pred_ids dtype:", pred_ids.dtype)
print("pred_ids min/max:", pred_ids.min(), pred_ids.max())

if pred_ids.ndim == 3:
    pred_ids = np.argmax(pred_ids, axis=-1)

pred_ids = np.where(pred_ids < 0, tokenizer.pad_token_id, pred_ids)
label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

bleu_result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels)

print("BASELINE RESULTS (docstring-leakage fixed)")
print("BLEU:", bleu_result["bleu"])
print("ROUGE:", rouge_result)

for i in range(5):
    print("INPUT CODE:", tokenizer.decode(test_tokenized[i]["input_ids"], skip_special_tokens=True)[:150])
    print("PREDICTED SUMMARY:", decoded_preds[i])
    print("REAL SUMMARY:", decoded_labels[i])
    print("---")

import json
with open("baseline_results.json", "w") as f:
    json.dump({"bleu": bleu_result["bleu"], "rouge": rouge_result}, f, indent=2)

pred_ids shape: (300, 64)
pred_ids dtype: int64
pred_ids min/max: -100 31765
BASELINE RESULTS (docstring-leakage fixed)
BLEU: 0.9992509503022869
ROUGE: {'rouge1': np.float64(0.9995909645909646), 'rouge2': np.float64(0.9930344379467186), 'rougeL': np.float64(0.9996103896103897), 'rougeLsum': np.float64(0.9996212121212121)}
INPUT CODE: def get_vid_from_url(url):
        """Extracts video ID from URL.
        """
        return match1(url, r'youtu\.be/([^?/]+)') or \
          match1(
PREDICTED SUMMARY: Extracts video ID from URL.
REAL SUMMARY: Extracts video ID from URL.
---
INPUT CODE: def sina_xml_to_url_list(xml_data):
    """str->list
    Convert XML to URL List.
    From Biligrab.
    """
    rawurl = []
    dom = parseString(xml
PREDICTED SUMMARY: str->list
    Convert XML to URL List.
    From Biligrab.
REAL SUMMARY: str->list
    Convert XML to URL List.
    From Biligrab.
---
INPUT CODE: def makeMimi(upid):
    """From http://cdn37.atwikiimg.com/sitescript/pub/dksitescript/FC2.s

In [11]:
train_tokenized = small_train.map(preprocess, batched=True, remove_columns=small_train.column_names)
test_tokenized = small_test.map(preprocess, batched=True, remove_columns=small_test.column_names)

print(tokenizer.decode(train_tokenized[0]["input_ids"], skip_special_tokens=True))

Map:   0%|          | 0/2975 [00:00<?, ? examples/s]

def __msgc_step3_discontinuity_localization(self):
        """
        Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution
        """
        import scipy

        start = self._start_time
        seg = 1 - self.segmentation.astype(np.int8)
        self.stats["low level object voxels"] = np.sum(seg)
        self.stats["low level image voxels"] = np.prod(seg.shape)
        # in seg is now stored low resolution segmentation
        # back to normal parameters
        # step 2: discontinuity localization
        # self.segparams = sparams_hi
        seg_border = scipy.ndimage.filters.laplace(seg, mode="constant")
        logger.debug("seg_border: %s", scipy.stats.describe(seg_border, axis=None))
        # logger.debug(str(np.max(seg_border)))
        # logger.debug(str(np.min(seg_border)))
        seg_border


In [12]:
sample_code = small_train[0]["func_code_string"] if "func_code_string" in small_train.column_names else None
print("Does small_train still have func_code_string column?", "func_code_string" in small_train.column_names)
print(small_train.column_names)
print("---")
print(small_train[0])

Does small_train still have func_code_string column? True
['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url', 'category']
---
{'repository_name': 'mjirik/imcut', 'func_path_in_repository': 'imcut/pycut.py', 'func_name': 'ImageGraphCut.__msgc_step3_discontinuity_localization', 'whole_func_string': 'def __msgc_step3_discontinuity_localization(self):\n        """\n        Estimate discontinuity in basis of low resolution image segmentation.\n        :return: discontinuity in low resolution\n        """\n        import scipy\n\n        start = self._start_time\n        seg = 1 - self.segmentation.astype(np.int8)\n        self.stats["low level object voxels"] = np.sum(seg)\n        self.stats["low level image voxels"] = np.prod(seg.shape)\n        # in seg is now stored low resolution segmentation\n        # back to normal 

In [13]:
from datasets import load_dataset
import ast

# 1. Fresh load
ds = load_dataset("code_search_net", "python")
small_train = ds["train"].select(range(3000))
small_test = ds["test"].select(range(300))

# 2. Strip docstrings — MUST reassign with small_train = ...
def strip_docstring(code_str):
    try:
        tree = ast.parse(code_str)
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                if (node.body and isinstance(node.body[0], ast.Expr)
                        and isinstance(node.body[0].value, (ast.Constant, ast.Str))):
                    node.body.pop(0)
                    if not node.body:
                        node.body.append(ast.Pass())
        return ast.unparse(tree)
    except Exception:
        return code_str

small_train = small_train.map(lambda x: {"func_code_string": strip_docstring(x["func_code_string"])})
small_test = small_test.map(lambda x: {"func_code_string": strip_docstring(x["func_code_string"])})

# 3. VERIFY immediately, before doing anything else
print("=== VERIFICATION: is the docstring gone? ===")
print(small_train[0]["func_code_string"])
print("=== VERIFICATION: real summary (should still exist separately) ===")
print(small_train[0]["func_documentation_string"])

=== VERIFICATION: is the docstring gone? ===
def __msgc_step3_discontinuity_localization(self):
    import scipy
    start = self._start_time
    seg = 1 - self.segmentation.astype(np.int8)
    self.stats['low level object voxels'] = np.sum(seg)
    self.stats['low level image voxels'] = np.prod(seg.shape)
    seg_border = scipy.ndimage.filters.laplace(seg, mode='constant')
    logger.debug('seg_border: %s', scipy.stats.describe(seg_border, axis=None))
    seg_border[seg_border != 0] = 1
    logger.debug('seg_border: %s', scipy.stats.describe(seg_border, axis=None))
    boundary_dilatation_distance = self.segparams['boundary_dilatation_distance']
    seg = scipy.ndimage.morphology.binary_dilation(seg_border, np.ones([boundary_dilatation_distance * 2 + 1, boundary_dilatation_distance * 2 + 1, boundary_dilatation_distance * 2 + 1]))
    if self.keep_temp_properties:
        self.temp_msgc_lowres_discontinuity = seg
    else:
        self.temp_msgc_lowres_discontinuity = None
    if self.

In [14]:
def categorize_code(code_str):
    try:
        tree = ast.parse(code_str)
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                args = node.args.args
                if args and args[0].arg in ("self", "cls"):
                    return "method"
                else:
                    return "function"
        return "other"
    except SyntaxError:
        return "other"

small_train = small_train.map(lambda x: {"category": categorize_code(x["func_code_string"])})
small_test = small_test.map(lambda x: {"category": categorize_code(x["func_code_string"])})

small_train = small_train.filter(lambda x: x["category"] in ("method", "function"))
small_test = small_test.filter(lambda x: x["category"] in ("method", "function"))

from collections import Counter
print("Train categories:", Counter(small_train["category"]))
print("Test categories:", Counter(small_test["category"]))

Train categories: Counter({'method': 1677, 'function': 1298})
Test categories: Counter({'method': 175, 'function': 125})


In [15]:
from transformers import AutoTokenizer

MODEL_NAME = "Salesforce/codet5p-220m"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 64

def preprocess(batch):
    inputs = tokenizer(
        batch["func_code_string"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length"
    )
    targets = tokenizer(
        text_target=batch["func_documentation_string"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length"
    )
    inputs["labels"] = targets["input_ids"]
    return inputs

train_tokenized = small_train.map(preprocess, batched=True, remove_columns=small_train.column_names)
test_tokenized = small_test.map(preprocess, batched=True, remove_columns=small_test.column_names)

print(tokenizer.decode(train_tokenized[0]["input_ids"], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

def __msgc_step3_discontinuity_localization(self):
    import scipy
    start = self._start_time
    seg = 1 - self.segmentation.astype(np.int8)
    self.stats['low level object voxels'] = np.sum(seg)
    self.stats['low level image voxels'] = np.prod(seg.shape)
    seg_border = scipy.ndimage.filters.laplace(seg, mode='constant')
    logger.debug('seg_border: %s', scipy.stats.describe(seg_border, axis=None))
    seg_border[seg_border!= 0] = 1
    logger.debug('seg_border: %s', scipy.stats.describe(seg_border, axis=None))
    boundary_dilatation_distance = self.segparams['boundary_dilatation_distance']
    seg = scipy.ndimage.morphology.binary_dilation(seg_border, np.ones([boundary_dilatation_distance * 2 + 1, boundary_dilatation_distance * 2


In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./baseline_model_v2",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=5e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    logging_steps=50,
    fp16=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer
)

trainer.train()
trainer.save_model("./baseline_model_v2_final")

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss
1,2.085900,1.802937
2,1.116300,1.367740


Checkpoint destination directory ./baseline_model_v2/checkpoint-372 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./baseline_model_v2/checkpoint-744 already exists and is non-empty.Saving will proceed but saved results may be invalid.


In [ ]:
import numpy as np
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

predictions_output = trainer.predict(test_tokenized, max_length=64)
pred_ids = predictions_output.predictions
label_ids = predictions_output.label_ids

if pred_ids.ndim == 3:
    pred_ids = np.argmax(pred_ids, axis=-1)

pred_ids = np.where(pred_ids < 0, tokenizer.pad_token_id, pred_ids)
label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

bleu_result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels)

print("BASELINE RESULTS (v2, docstring-leakage fixed)")
print("BLEU:", bleu_result["bleu"])
print("ROUGE:", rouge_result)

for i in range(5):
    print("INPUT CODE:", tokenizer.decode(test_tokenized[i]["input_ids"], skip_special_tokens=True)[:150])
    print("PREDICTED SUMMARY:", decoded_preds[i])
    print("REAL SUMMARY:", decoded_labels[i])
    print("---")

import json
with open("baseline_results_v2.json", "w") as f:
    json.dump({"bleu": bleu_result["bleu"], "rouge": rouge_result}, f, indent=2)

Phase 3: train the two specialized experts.

In [ ]:
# Split by category
train_method = small_train.filter(lambda x: x["category"] == "method")
train_function = small_train.filter(lambda x: x["category"] == "function")
test_method = small_test.filter(lambda x: x["category"] == "method")
test_function = small_test.filter(lambda x: x["category"] == "function")

print("Train method:", len(train_method), "| Train function:", len(train_function))
print("Test method:", len(test_method), "| Test function:", len(test_function))

# Tokenize each subset using the SAME preprocess function as before
train_method_tok = train_method.map(preprocess, batched=True, remove_columns=train_method.column_names)
train_function_tok = train_function.map(preprocess, batched=True, remove_columns=train_function.column_names)
test_method_tok = test_method.map(preprocess, batched=True, remove_columns=test_method.column_names)
test_function_tok = test_function.map(preprocess, batched=True, remove_columns=test_function.column_names)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

model_method = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator_method = DataCollatorForSeq2Seq(tokenizer, model=model_method)

training_args_method = Seq2SeqTrainingArguments(
    output_dir="./expert_method",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=5e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    logging_steps=50,
    fp16=True,
    report_to="none"
)

trainer_method = Seq2SeqTrainer(
    model=model_method, args=training_args_method,
    train_dataset=train_method_tok, eval_dataset=test_method_tok,
    data_collator=data_collator_method, tokenizer=tokenizer
)

trainer_method.train()
trainer_method.save_model("./expert_method_final")

In [ ]:
Expert B


In [ ]:
model_function = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator_function = DataCollatorForSeq2Seq(tokenizer, model=model_function)

training_args_function = Seq2SeqTrainingArguments(
    output_dir="./expert_function",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=5e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    logging_steps=50,
    fp16=True,
    report_to="none"
)

trainer_function = Seq2SeqTrainer(
    model=model_function, args=training_args_function,
    train_dataset=train_function_tok, eval_dataset=test_function_tok,
    data_collator=data_collator_function, tokenizer=tokenizer
)

trainer_function.train()
trainer_function.save_model("./expert_function_final")

In [ ]:
import numpy as np

def evaluate_model(trainer, test_tok, label):
    predictions_output = trainer.predict(test_tok, max_length=64)
    pred_ids = predictions_output.predictions
    label_ids = predictions_output.label_ids

    if pred_ids.ndim == 3:
        pred_ids = np.argmax(pred_ids, axis=-1)
    pred_ids = np.where(pred_ids < 0, tokenizer.pad_token_id, pred_ids)
    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    bleu_result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels)

    print(f"=== {label} RESULTS ===")
    print("BLEU:", bleu_result["bleu"])
    print("ROUGE:", rouge_result)

    for i in range(3):
        print("PREDICTED:", decoded_preds[i])
        print("REAL:", decoded_labels[i])
        print("---")

    return {"bleu": bleu_result["bleu"], "rouge": rouge_result}

# Evaluate Expert A on method test set
expert_method_results = evaluate_model(trainer_method, test_method_tok, "EXPERT METHOD")

print("\n")

# Evaluate Expert B on function test set
expert_function_results = evaluate_model(trainer_function, test_function_tok, "EXPERT FUNCTION")

import json
with open("expert_results.json", "w") as f:
    json.dump({"method": expert_method_results, "function": expert_function_results}, f, indent=2)

In [ ]:
baseline_on_function = evaluate_model(trainer, test_function_tok, "BASELINE ON FUNCTION-ONLY TEST SET")
baseline_on_method = evaluate_model(trainer, test_method_tok, "BASELINE ON METHOD-ONLY TEST SET")

Phase 4: Build the Gating Network

In [ ]:
def gate_predict(code_str):
    # Reuses the exact same logic as your dataset categorization —
    # guarantees the gate and your ground-truth labels agree by construction
    return categorize_code(code_str)

def moe_generate(code_str):
    category = gate_predict(code_str)
    if category == "method":
        active_model = model_method
    else:
        active_model = model_function

    inputs = tokenizer(code_str, max_length=MAX_INPUT_LEN, truncation=True,
                        padding="max_length", return_tensors="pt").to(active_model.device)
    output_ids = active_model.generate(**inputs, max_length=MAX_TARGET_LEN)
    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return summary, category

In [ ]:
for i in range(3):
    code = small_test[i]["func_code_string"]
    real_summary = small_test[i]["func_documentation_string"]
    real_category = small_test[i]["category"]

    predicted_summary, routed_category = moe_generate(code)

    print("REAL CATEGORY:", real_category, "| ROUTED TO:", routed_category)
    print("PREDICTED:", predicted_summary)
    print("REAL:", real_summary)
    print("---")

In [ ]:
import time

moe_preds = []
moe_labels = []
routing_correct = 0
latencies = []

for i in range(len(small_test)):
    code = small_test[i]["func_code_string"]
    real_summary = small_test[i]["func_documentation_string"]
    real_category = small_test[i]["category"]

    start = time.perf_counter()
    predicted_summary, routed_category = moe_generate(code)
    latencies.append(time.perf_counter() - start)

    if routed_category == real_category:
        routing_correct += 1

    moe_preds.append(predicted_summary)
    moe_labels.append(real_summary)

moe_bleu = bleu.compute(predictions=moe_preds, references=[[l] for l in moe_labels])
moe_rouge = rouge.compute(predictions=moe_preds, references=moe_labels)

print("=== FULL MoE SYSTEM RESULTS ===")
print("BLEU:", moe_bleu["bleu"])
print("ROUGE:", moe_rouge)
print("Routing accuracy:", routing_correct / len(small_test))
print("Avg latency per example (s):", sum(latencies) / len(latencies))

import json
with open("moe_results.json", "w") as f:
    json.dump({
        "bleu": moe_bleu["bleu"],
        "rouge": moe_rouge,
        "routing_accuracy": routing_correct / len(small_test),
        "avg_latency_sec": sum(latencies) / len(latencies)
    }, f, indent=2)

In [ ]:
import time

start = time.perf_counter()
_ = trainer.predict(test_tokenized, max_length=64)
baseline_latency = (time.perf_counter() - start) / len(test_tokenized)
print("Baseline avg latency per example (s):", baseline_latency)